# Quickly analyse a given map

In [ ]:
import plotly


In [10]:
# %% [markdown]
# # NetCDF Analysis Notebook with Widgets
#
# **Features:**
# - Select a predefined folder.
# - Toggle between single-file or multi-file analysis.
# - Use a **slider** to select a time slice.
# - Use a **dropdown** to choose which analysis plot to display.
# - Visualize results interactively.

# %%
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import linregress
from IPython.display import display, clear_output
import ipywidgets as widgets
import glob
import os

# %% [markdown]
# ## 1. Setup Widgets

# %%
# Predefined folders (update this list with your folder paths)
predefined_folders = {
    "Lowres": "/work/uo1075/u244085/trees_in_the_city/data/era5_land/t2max/lowres",
    "Highres": "/work/uo1075/u244085/trees_in_the_city/data/era5_land/t2max/highres",
    "Mask": "/work/uo1075/u244085/trees_in_the_city/data/era5_land/t2max/lowres_in_highres_grid",
}

# Widget to select a predefined folder
folder_dropdown = widgets.Dropdown(
    description='Select Folder:',
    options=list(predefined_folders.keys()),
    disabled=False,
)

# Widget to toggle between single/multi-file mode
mode_toggle = widgets.ToggleButton(
    value=False,
    description='Multi-File Mode',
    disabled=False,
    button_style='',
    tooltip='Toggle to switch between single-file and multi-file analysis',
)

# Widget to select single file
single_file_dropdown = widgets.Dropdown(
    description='Single File:',
    disabled=False,
    options=[],
)

# Widget to select multiple files
multi_file_selector = widgets.SelectMultiple(
    description='Multi-File:',
    disabled=False,
    options=[],
    value=[],
)

# Widget to select time slice (slider)
time_slider = widgets.IntSlider(
    description='Time Index:',
    min=0,
    max=0,  # Will update dynamically
    step=1,
    value=0,
)

# Widget to select analysis plot
plot_dropdown = widgets.Dropdown(
    description='Analysis Plot:',
    options=[
        "Time Slice",
        "Mean Over Time",
        "Trend Analysis",
        "File Comparison (Multi-File Only)",
    ],
    disabled=False,
)

# Widget to trigger analysis
analyze_button = widgets.Button(
    description="Run Analysis",
    disabled=False,
    button_style='',
    tooltip='Click to run the analysis',
)

# Output area for results
output = widgets.Output()

# %%
# Function to update file selectors and time slider based on the selected folder
def update_file_selectors(change):
    selected_folder = predefined_folders[folder_dropdown.value]
    nc_files = glob.glob(os.path.join(selected_folder, "*.nc"))

    if mode_toggle.value:  # Multi-file mode
        single_file_dropdown.options = nc_files
        single_file_dropdown.layout.display = 'none'
        multi_file_selector.options = nc_files
        multi_file_selector.layout.display = 'flex'
    else:  # Single-file mode
        single_file_dropdown.options = nc_files
        single_file_dropdown.layout.display = 'flex'
        multi_file_selector.layout.display = 'none'

    # Update time slider max value if a dataset is loaded
    if single_file_dropdown.value:
        with xr.open_dataset(single_file_dropdown.value) as ds:
            time_slider.max = len(ds["valid_time"]) - 1
    elif len(multi_file_selector.value) > 0:
        with xr.open_mfdataset(multi_file_selector.value, combine="by_coords") as ds:
            time_slider.max = len(ds["valid_time"]) - 1

# Function to update file selectors when the folder changes
def on_folder_change(change):
    update_file_selectors(change)

# Function to update file selectors when the mode changes
def on_mode_change(change):
    update_file_selectors(change)

# Function to update time slider when a file is selected
def on_file_change(change):
    if mode_toggle.value:  # Multi-file mode
        if len(multi_file_selector.value) > 0:
            with xr.open_mfdataset(multi_file_selector.value, combine="by_coords") as ds:
                time_slider.max = len(ds["valid_time"]) - 1
    else:  # Single-file mode
        if single_file_dropdown.value:
            with xr.open_dataset(single_file_dropdown.value) as ds:
                time_slider.max = len(ds["valid_time"]) - 1

# Link the folder dropdown, mode toggle, and file selectors to update functions
folder_dropdown.observe(on_folder_change, names='value')
mode_toggle.observe(on_mode_change, names='value')
single_file_dropdown.observe(on_file_change, names='value')
multi_file_selector.observe(on_file_change, names='value')

# Initially hide the multi-file selector
multi_file_selector.layout.display = 'none'

# Display widgets
display(
    widgets.VBox([
        folder_dropdown,
        mode_toggle,
        single_file_dropdown,
        multi_file_selector,
        time_slider,
        plot_dropdown,
        analyze_button,
        output,
    ])
)

# %%
# Define the analysis function
def run_analysis(b):
    with output:
        clear_output()
        print("Running analysis...")

        selected_folder = predefined_folders[folder_dropdown.value]

        # Load data based on mode
        if mode_toggle.value:  # Multi-file mode
            if len(multi_file_selector.value) < 1:
                print("Error: Select at least 1 file for analysis.")
                return
            print(f"Loading {len(multi_file_selector.value)} files from {selected_folder}...")
            ds = xr.open_mfdataset(multi_file_selector.value, combine="by_coords", chunks={"valid_time": 10})
        else:  # Single-file mode
            if not single_file_dropdown.value:
                print("Error: Select a file for single-file analysis.")
                return
            print(f"Loading {single_file_dropdown.value}...")
            ds = xr.open_dataset(single_file_dropdown.value, chunks={"valid_time": 10})

        # Get the selected time slice
        time_index = time_slider.value
        if time_index >= len(ds["valid_time"]):
            time_index = 0
        time_slice = ds["valid_time"][time_index].values

        # %% Generate the selected plot
        if plot_dropdown.value == "Time Slice":
            t2m_slice = ds["t2m"].sel(valid_time=time_slice)
            plt.figure(figsize=(10, 6))
            t2m_slice.plot()
            plt.title(f"Temperature at {time_slice}")
            plt.show()

        elif plot_dropdown.value == "Mean Over Time":
            mean_t2m = ds["t2m"].mean(dim="valid_time")
            plt.figure(figsize=(10, 6))
            mean_t2m.plot()
            plt.title("Mean Temperature Over Latitude/Longitude")
            plt.show()

        elif plot_dropdown.value == "Trend Analysis":
            def compute_trend(data):
                x = data["valid_time"].astype("int64").values
                y = data["t2m"].values
                slope, _, _, _, _ = linregress(x, y)
                return slope

            trend = xr.apply_ufunc(
                compute_trend,
                ds,
                input_core_dims=[["valid_time"]],
                output_core_dims=[[]],
                vectorize=True,
            )
            plt.figure(figsize=(10, 6))
            trend.plot()
            plt.title("Temperature Trend (Slope of Linear Regression)")
            plt.show()

        elif plot_dropdown.value == "File Comparison (Multi-File Only)":
            if mode_toggle.value and len(multi_file_selector.value) >= 2:
                print("Comparing files...")
                ds1 = xr.open_dataset(multi_file_selector.value[0], chunks={"valid_time": 10})
                ds2 = xr.open_dataset(multi_file_selector.value[1], chunks={"valid_time": 10})

                diff = ds1["t2m"] - ds2["t2m"]
                diff_mean = diff.mean(dim="valid_time")

                plt.figure(figsize=(10, 6))
                diff_mean.plot()
                plt.title("Mean Temperature Difference (File1 - File2)")
                plt.show()
            else:
                print("Error: File comparison requires multi-file mode and at least 2 files.")

        print("Analysis complete!")

# Link the analyze button to the function
analyze_button.on_click(run_analysis)

# %% [markdown]
# ## 2. Instructions
#
# 1. **Select Folder**: Choose a predefined folder from the dropdown.
# 2. **Toggle Mode**: Use the `Multi-File Mode` button to switch between single-file and multi-file analysis.
# 3. **Select Files**:
#    - In **single-file mode**, select a file from the dropdown.
#    - In **multi-file mode**, select at least 1 file from the multi-select widget.
# 4. **Select Time Slice**: Use the slider to choose a time index.
# 5. **Select Plot**: Use the dropdown to choose which analysis plot to display.
# 6. **Run Analysis**: Click the `Run Analysis` button to execute the workflow.
#
# The results will appear below the button.